# S6_7 — Honest Failure Analysis

**Leaf-JEPA IRP** | Stage 6 — Analysis and Interpretation


**Purpose:** Identify, investigate, and document systematic model failures.
This notebook synthesises ALL previous Stage 6 outputs to produce the failure analysis
section of Chapter 5 (Discussion).

**Why this matters:** Examiners reward honest, analytical treatment of failures far more
than hiding them. A student who presents perfect results is either cherry-picking or
not looking hard enough.

**Outputs:**
- `failure_summary.json` — structured failure analysis
- `failure_examples_grid.png` — misclassified images with annotations
- `method_reliability.csv` — variance and convergence statistics per method

**This notebook should run LAST — it synthesises all other Stage 6 outputs.**

## Initialization

In [1]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage6_analysis_and_interpretation.config_stage6 import *
from stage6_analysis_and_interpretation.analysis_utils import (
    set_seed, load_json, save_json, load_split_data, print_section,
)

import matplotlib.pyplot as plt
from PIL import Image

set_seed(RANDOM_SEED)
STAGE6_FIGURES.mkdir(parents=True, exist_ok=True)
STAGE6_TABLES.mkdir(parents=True, exist_ok=True)


## Universally Hard Classes

In [ ]:
print_section("F1: Universally Hard Disease Classes")

hard_path = STAGE6_TABLES / "hard_classes_analysis.csv"
if hard_path.exists():
    hard_df = pd.read_csv(hard_path, index_col=0)
    bottom = hard_df.head(8)

    print("  8 hardest classes across ALL methods:")
    for cls_name, row in bottom.iterrows():
        print(f"    {cls_name}: avg F1 = {row['avg_f1']:.4f} (std = {row['std_f1']:.4f})")

    # Hypothesis generation
    print("\n  Hypothesised difficulty factors:")
    # Load Stage 2 data if available
    class_counts_path = ANALYSIS_DIR / "class_distribution.json" if ANALYSIS_DIR else None
    bg_path = ANALYSIS_DIR / "background_ratios.json" if ANALYSIS_DIR else None

    for cls_name in bottom.index:
        factors = []

        # Check if rare
        if class_counts_path and class_counts_path.exists():
            counts = load_json(class_counts_path)
            if isinstance(counts, dict) and cls_name in counts:
                if counts[cls_name] < 500:
                    factors.append(f"rare class ({counts[cls_name]} samples)")

        # Check if high background
        if bg_path and bg_path.exists():
            bg_data = load_json(bg_path)
            if isinstance(bg_data, dict) and cls_name in bg_data:
                if bg_data[cls_name] > 0.75:
                    factors.append(f"high background ({bg_data[cls_name]:.0%})")

        # Check if in top confused pairs
        persist_path = STAGE6_TABLES / "confusion_pair_persistence.csv"
        if persist_path.exists():
            persist_df = pd.read_csv(persist_path)
            confusable_with = persist_df[persist_df["true_class"] == cls_name]
            if not confusable_with.empty:
                top_conf = confusable_with.iloc[0]
                factors.append(f"confused with {top_conf['pred_class']} ({top_conf['avg_confusion']:.3f})")

        factor_str = "; ".join(factors) if factors else "investigate manually"
        print(f"    {cls_name}: {factor_str}")
else:
    print("  \u26a0\ufe0f Run S6_4 first to generate hard_classes_analysis.csv")

## F2 — PEFT Method Reliability

In [ ]:
print_section("F2: PEFT Method Reliability")

reliability = []

s1_path = PEFT_RESULTS_DIR / "S1_method_comparison_summary.json"
if s1_path.exists():
    s1 = load_json(s1_path)
    if isinstance(s1, dict):
        for method, data in s1.items():
            if not isinstance(data, dict):
                continue
            entry = {"method": method}

            # Variance across seeds
            f1_std = data.get("macro_f1_std", None)
            if f1_std is not None:
                entry["f1_std"] = f1_std
                entry["high_variance"] = f1_std > 0.02

            # Convergence issues
            entry["converged"] = data.get("converged", True)

            # Training time
            entry["train_time_per_epoch"] = data.get("train_time_per_epoch", None)

            reliability.append(entry)

    if reliability:
        rel_df = pd.DataFrame(reliability)
        rel_df.to_csv(STAGE6_TABLES / "method_reliability.csv", index=False)

        print("  Method Reliability Summary:")
        print(rel_df.to_string(index=False))

        # Flag unreliable methods
        unreliable = [r for r in reliability if r.get("high_variance") or not r.get("converged")]
        if unreliable:
            print(f"\n  \u26a0\ufe0f {len(unreliable)} methods flagged as unreliable:")
            for r in unreliable:
                reasons = []
                if r.get("high_variance"):
                    reasons.append(f"high variance (std={r['f1_std']:.4f})")
                if not r.get("converged"):
                    reasons.append("convergence failure")
                print(f"    {r['method']}: {', '.join(reasons)}")
        else:
            print("\n  All methods appear reliable.")
else:
    print("  \u26a0\ufe0f S1 results not found — cannot assess reliability")

## F3 — Cross-Domain Transfer Failures

In [ ]:
print_section("F3: Cross-Domain Transfer Analysis")

s3_path = PEFT_RESULTS_DIR / "S3_cross_domain_results.json"
if s3_path.exists():
    s3 = load_json(s3_path)

    print("  PlantVillage -> PlantDoc transfer performance:")
    transfer_gaps = []

    if isinstance(s3, dict):
        for method, data in s3.items():
            if not isinstance(data, dict):
                continue
            pv_f1 = data.get("plantvillage_f1", data.get("source_f1", None))
            pd_f1 = data.get("plantdoc_f1", data.get("target_f1", None))
            if pv_f1 is not None and pd_f1 is not None:
                gap = pv_f1 - pd_f1
                transfer_gaps.append({
                    "method": method,
                    "plantvillage_f1": pv_f1,
                    "plantdoc_f1": pd_f1,
                    "domain_gap": gap,
                })
                print(f"    {method}: PV={pv_f1:.4f}, PD={pd_f1:.4f}, gap={gap:.4f}")

    if transfer_gaps:
        gap_df = pd.DataFrame(transfer_gaps).sort_values("domain_gap")
        gap_df.to_csv(STAGE6_TABLES / "cross_domain_gaps.csv", index=False)

        best_transfer = gap_df.iloc[0]
        worst_transfer = gap_df.iloc[-1]
        print(f"\n  Best transfer: {best_transfer['method']} (gap = {best_transfer['domain_gap']:.4f})")
        print(f"  Worst transfer: {worst_transfer['method']} (gap = {worst_transfer['domain_gap']:.4f})")
else:
    print("  \u26a0\ufe0f S3 cross-domain results not found")
    print("  Run Stage 5 S3 first, or note this as a limitation")

## F4 — Where Did Leaf-JEPA Underperform?

In [ ]:
print_section("F4: Hypothesis Testing — Where Expectations Weren't Met")

failure_findings = []

# Check RQ1: B5 > B3?
b3_path = BASELINES_DIR / "B3_aggregate.json"
b5_path = BASELINES_DIR / "B5_aggregate.json"
if b3_path.exists() and b5_path.exists():
    b3 = load_json(b3_path)
    b5 = load_json(b5_path)
    b3_f1 = b3.get("macro_f1_mean", b3.get("macro_f1", 0))
    b5_f1 = b5.get("macro_f1_mean", b5.get("macro_f1", 0))
    delta = b5_f1 - b3_f1

    if delta < 0.01:
        failure_findings.append({
            "rq": "RQ1",
            "hypothesis": "Leaf-JEPA LP > Generic I-JEPA LP",
            "finding": f"Delta was only {delta:+.4f} — smaller than expected",
            "explanation": "PlantVillage images may be visually simple enough that "
                           "ImageNet features already capture relevant patterns. "
                           "The lab-controlled backgrounds reduce the need for "
                           "domain-specific representations.",
            "future_work": "Pretrain on larger, more diverse plant disease datasets "
                           "(CGIAR, iNaturalist plant subset) to increase domain shift."
        })
        print(f"  RQ1: B5 - B3 = {delta:+.4f} (BELOW expected improvement)")
    else:
        print(f"  RQ1: B5 - B3 = {delta:+.4f} (hypothesis confirmed)")

## is RQ5: PEFT > CNNs at low labels?

In [ ]:
crossover_path = STAGE6_DATA / "crossover_analysis.json"
if crossover_path.exists():
    crossover = load_json(crossover_path)
    if not crossover:
        failure_findings.append({
            "rq": "RQ5",
            "hypothesis": "Leaf-JEPA + PEFT beats supervised CNNs at <=10% labels",
            "finding": "No crossover found — CNNs competitive across all fractions",
            "explanation": "PlantVillage's visual simplicity and clear backgrounds "
                           "allow supervised CNNs to learn effectively even from few samples. "
                           "The SSL pretraining advantage is smaller on easy datasets.",
            "future_work": "Test on more challenging field-condition datasets where "
                           "background variation and image quality create a domain gap."
        })
        print(f"  RQ5: No crossover found — PEFT did not surpass CNNs at low labels")
    else:
        for key, frac in crossover.items():
            print(f"  RQ5: {key} crossover at {frac:.1%}")

## Check A3: masking contribution

In [ ]:
a3_path = STAGE6_DATA / "ablation3_masking_strategy.json"
if a3_path.exists():
    a3 = load_json(a3_path)
    for comp in a3.get("comparisons", []):
        if comp.get("delta", 0) < 0.005:
            failure_findings.append({
                "rq": "Novel contribution",
                "hypothesis": "Disease-biased masking improves representations",
                "finding": f"Delta was only {comp.get('delta', 0):+.4f}",
                "explanation": "The hue-deviation saliency map may not accurately "
                               "capture disease regions for all pathogen types. "
                               "Viral diseases (mosaic patterns) span the entire leaf, "
                               "making region-biased masking less effective.",
                "future_work": "Use learned saliency (Grad-CAM from a warm-up classifier) "
                               "instead of hand-crafted hue deviation."
            })

## Compile failure summary and example grid

In [ ]:
print_section("Failure Summary")

# Print all findings in the three-sentence pattern
for f in failure_findings:
    print(f"\n  [{f['rq']}]")
    print(f"  FINDING:     Contrary to hypothesis '{f['hypothesis']}', {f['finding']}")
    print(f"  EXPLANATION: {f['explanation']}")
    print(f"  FUTURE WORK: {f['future_work']}")

save_json(failure_findings, STAGE6_DATA / "failure_summary.json")

# Generate failure examples grid — show misclassified images
print("\n  Generating failure examples grid...")
image_paths, labels, class_names = load_split_data(SPLITS_DIR, split="test")

# Try to load predictions from best model to find misclassified images
pred_files = list(PEFT_RESULTS_DIR.glob("*_predictions.json")) + \
             list(BASELINES_DIR.glob("*_predictions.json"))

misclassified = []
if pred_files:
    preds = load_json(pred_files[0])
    y_true = preds.get("y_true", [])
    y_pred = preds.get("y_pred", [])
    for i, (t, p) in enumerate(zip(y_true, y_pred)):
        if t != p and i < len(image_paths):
            misclassified.append((i, t, p))

    if misclassified:
        # Show 12 random misclassified examples
        rng = np.random.RandomState(RANDOM_SEED)
        sample = rng.choice(len(misclassified), min(12, len(misclassified)), replace=False)

        fig, axes = plt.subplots(3, 4, figsize=(16, 12))
        for ax_idx, mc_idx in enumerate(sample):
            if ax_idx >= 12:
                break
            img_idx, true_label, pred_label = misclassified[mc_idx]
            img = Image.open(image_paths[img_idx]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
            ax = axes[ax_idx // 4, ax_idx % 4]
            ax.imshow(np.array(img))
            true_name = class_names[true_label] if true_label < len(class_names) else str(true_label)
            pred_name = class_names[pred_label] if pred_label < len(class_names) else str(pred_label)
            ax.set_title(f"True: {true_name}\nPred: {pred_name}", fontsize=7, color="red")
            ax.axis("off")

        # Hide unused axes
        for ax_idx in range(len(sample), 12):
            axes[ax_idx // 4, ax_idx % 4].axis("off")

        plt.suptitle("Misclassified Examples", fontsize=14)
        plt.tight_layout()
        fig.savefig(STAGE6_FIGURES / "failure_examples_grid.png")
        plt.close(fig)
        print(f"  Failure examples grid saved ({len(misclassified)} total misclassified)")
else:
    print("  \u26a0\ufe0f No prediction files found — cannot generate failure grid")
    print("  Save predictions as JSON in Stage 3/5 notebooks: ")
    print('    json.dump({"y_true": [...], "y_pred": [...]}, open("..._predictions.json", "w"))')

## Final Synthesis

In [ ]:
print_section("Stage 6 Complete — Final Synthesis")

print("  DISSERTATION WRITING GUIDE:")
print("  " + "-"*56)
print()
print("  Chapter 4 (Results) sections from Stage 6:")
print("    4.1 Statistical significance → S6_1 outputs")
print("    4.2 Pareto frontier          → S6_5 pareto_frontier.png")
print("    4.3 t-SNE progression        → S6_2 tsne_progression_3panel.png")
print("    4.4 Label efficiency         → S6_5 label_efficiency_all_methods.png")
print("    4.5 Attention maps           → S6_3 attention_comparison_grid.png")
print("    4.6 Confusion analysis       → S6_4 confusion_diff figures")
print("    4.7 Ablations                → S6_6 ablation summary")
print("    4.8 Hard classes             → S6_4 hard_classes_analysis.csv")
print()
print("  Chapter 5 (Discussion) sections from Stage 6:")
print("    5.1-5.5 RQ answers           → cite specific S6 numbers")
print("    5.6 Novel contribution       → A3 masking ablation results")
print("    5.7 Limitations              → S6_1 (n=3 power), S6_7 findings")
print("    5.8 Failure analysis          → S6_7 failure_summary.json")
print("    5.9 Future work              → S6_7 future_work suggestions")

# Check all outputs
print("\n  \u2705 S6_7 complete — Stage 6 analysis finished.")
print("  Run S6_0 Cell 3 for the full definition-of-done checklist.")

## Critical Analysis

### The Three-Sentence Pattern
Every failure finding should follow this structure:
1. **FINDING:** "Contrary to hypothesis H[N], [specific result]."
2. **EXPLANATION:** "We hypothesise this occurred because [specific mechanism]."
3. **FUTURE WORK:** "Future work could address this by [concrete suggestion]."

This pattern demonstrates research maturity — the examiner sees that you understand
not just WHAT happened but WHY, and what to do about it.

### Honest vs Hiding
- Hiding failures: "Our model achieves state-of-the-art performance."
  → Examiner thinks: "On which subset? What about the hard classes?"
- Honest reporting: "While Leaf-JEPA improved representation quality for 30 of 38 classes,
  8 classes with high inter-class visual similarity showed minimal improvement."
  → Examiner thinks: "This student understands the problem space deeply."

### Stage 6 is Complete
All dissertation-ready figures and tables are now in the Stage 6 outputs directory.
Proceed to Stage 7 (Computational Efficiency) for VRAM/speed profiling, or directly
to Stage 8 (Dissertation Writing) if you already have profiling data from Stage 5.